<a href="https://colab.research.google.com/github/ANEESAAYAZ/internship_LinearRegression/blob/main/Internship_LinearRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("properties_features.csv",  parse_dates=["listed_date"])

In [3]:
print( f"loaded {df.shape[0]} rows x {df.shape[1]} columns")

loaded 3000 rows x 71 columns


In [4]:
FORBIDDEN = [ "price_per_sqft", "rental_yield_pct",
             "log_price_aed",  "area_price_encoded",
              "days_on_market",    "sold_fast", "listing_status",
              "annual_rent_aed",    "log_annual_rent_aed"]

In [5]:
print("\nexcluded because they would leak the answer:")
for col in FORBIDDEN:
  print(f" {col}")


excluded because they would leak the answer:
 price_per_sqft
 rental_yield_pct
 log_price_aed
 area_price_encoded
 days_on_market
 sold_fast
 listing_status
 annual_rent_aed
 log_annual_rent_aed


In [6]:
model_df = df[df["price_aed"].notna()].copy()

In [7]:
print(f"\nmodelling population: {len(model_df)} listings")


modelling population: 3000 listings


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
DROP = FORBIDDEN + ["listed_date", "description", "price_aed","latitude", "longitude"]
X = model_df.drop(columns=[c for c in DROP if c in model_df.columns])
y_aed = model_df["price_aed"]
y = np.log1p(y_aed)
print(f"\nfeature matrix: {X.shape[1]} columns")


feature matrix: 57 columns


In [10]:
print( f"target: log1p(price_aed)," f"skew {y_aed.skew():.2f} -> {y.skew():.2f}")

target: log1p(price_aed),skew 6.76 -> 0.51


In [11]:
X_train, X_test, y_train, y_test, aed_train, aed_test = train_test_split(  X, y, y_aed, test_size=0.2, random_state=42)

In [12]:
print(f"train {len(X_train)}   test {len(X_test)}")

train 2400   test 600


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [14]:
numeric_cols = X.select_dtypes( include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(    include=["object"]).columns.tolist()

print(f"\nnumeric: {len(numeric_cols)}   categorical: {len(categorical_cols)}")
print(f"categorical columns: {categorical_cols}")


numeric: 47   categorical: 10
categorical columns: ['emirate', 'area', 'property_type', 'completion_status', 'developer', 'ownership_type', 'furnishing', 'view', 'listed_by', 'agency_tier']


In [15]:
preprocessor = ColumnTransformer([("num", Pipeline([ ("impute", SimpleImputer(strategy="median")),
                                                    ("scale", StandardScaler())
                                                  ]), numeric_cols),
                                ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                                  ("encode", OneHotEncoder( handle_unknown="ignore", sparse_output=False,
                                                  min_frequency=10 ))
                                                ]), categorical_cols)
                                ])

In [16]:
from sklearn.linear_model import LinearRegression

In [17]:
model = Pipeline([("prep", preprocessor),("lm", LinearRegression())])
model.fit(X_train, y_train)
n_features = model.named_steps["prep"].transform(X_train).shape[1]

In [18]:
print(f"\nfitted on {len(X_train)} rows x {n_features} features after encoding")


fitted on 2400 rows x 188 features after encoding


In [19]:
pred_log = model.predict(X_test)
pred_aed = np.expm1(pred_log)

print("\n--- raw model output, before converting back ---")
print(f"{np.round(pred_log[:4], 2)}   <- these are LOGS, not prices")


--- raw model output, before converting back ---
[13.91 12.92 14.58 14.88]   <- these are LOGS, not prices


In [20]:
comparison = pd.DataFrame({ "actual_aed": aed_test.values,
                           "predicted_aed": pred_aed,}).assign( error_aed=lambda d: d["predicted_aed"] - d["actual_aed"],
                                                               error_pct=lambda d: 100 * (d["predicted_aed"] - d["actual_aed"]) / d["actual_aed"])

In [21]:
print("\n--- five test predictions, in dirhams ---")
print(comparison.head(5).round(0).to_string(index=False))


--- five test predictions, in dirhams ---
 actual_aed  predicted_aed  error_aed  error_pct
  1000000.0      1100715.0   100715.0       10.0
   400000.0       408367.0     8367.0        2.0
  1950000.0      2137537.0   187537.0       10.0
  3200000.0      2910028.0  -289972.0       -9.0
  2850000.0      2752667.0   -97333.0       -3.0


**Evaluation Metrics**

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [23]:
r2 = r2_score(y_test, pred_log)

In [24]:
rmse_log = np.sqrt(mean_squared_error(y_test, pred_log))

In [25]:
mae_aed = mean_absolute_error(aed_test, pred_aed)

In [26]:
medae_aed = np.median(np.abs(pred_aed - aed_test))
mape = np.mean(np.abs((pred_aed - aed_test) / aed_test)) * 100
within_10 = np.mean( np.abs((pred_aed - aed_test) / aed_test) <= 0.10) * 100
within_20 = np.mean(np.abs((pred_aed - aed_test) / aed_test) <= 0.20) * 100

print("\n--- in log space (what the model optimised) ---\n")
print(f"R2  {r2:.4f}")
print(f"RMSE {rmse_log:.4f} log units  (~{100 * (np.exp(rmse_log) - 1):.1f}% typical error)")

print("\n--- in dirhams (what the business understands) ---\n")
print(f"MAE AED {mae_aed:,.0f}")
print(f"median AED {medae_aed:,.0f}")
print(f"MAPE {mape:.1f}%")
print(f"within 10% of the true price: {within_10:.0f}% of listings")
print(f"within 20% of the true price: {within_20:.0f}% of listings")


--- in log space (what the model optimised) ---

R2  0.9456
RMSE 0.2279 log units  (~25.6% typical error)

--- in dirhams (what the business understands) ---

MAE AED 848,520
median AED 248,910
MAPE 17.4%
within 10% of the true price: 38% of listings
within 20% of the true price: 66% of listings


In [27]:
feature_names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series( model.named_steps["lm"].coef_,index=feature_names)

print("\n--- the 12 features that push price UP most ---")
for name, c in coefs.sort_values(ascending=False).head(12).items():
  print(f"{name:52s} {c:+.4f} ({100 * (np.exp(c) - 1):+8.1f}%)")

print("\n--- the 8 that push price DOWN most ---")
for name, c in coefs.sort_values().head(8).items():
  print(f"{name:52s} {c:+.4f} ({100 * (np.exp(c) - 1):+8.1f}%)")

GEO = [ "km_to_burj_khalifa","km_to_dxb_airport",
        "km_to_dubai_marina", "km_to_abu_dhabi_corniche",
        "distance_to_downtown_km"]

print("\n--- how correlated are the geo features with each other? ---")
print(X_train[GEO].corr().round(3).to_string())

print("\n--- and what did the model do with them? ---")
for n in GEO:
  print(f"num__{n:34s} {coefs['num__' + n]:+9.4f}")


--- the 12 features that push price UP most ---
num__km_to_dxb_airport                               +4.7798 (+11807.5%)
cat__emirate_Ras Al Khaimah                          +2.3062 (  +903.6%)
cat__area_Dubai Creek Harbour                        +2.0606 (  +685.1%)
cat__developer_RAK Properties                        +1.4464 (  +324.8%)
cat__area_Al Jaddaf                                  +1.3048 (  +268.7%)
num__km_to_dubai_marina                              +1.2703 (  +256.2%)
cat__area_Culture Village                            +1.2215 (  +239.2%)
cat__emirate_Sharjah                                 +1.1818 (  +226.0%)
cat__emirate_Ajman                                   +1.1234 (  +207.5%)
cat__area_Mina Al Arab                               +1.0904 (  +197.5%)
cat__area_Al Barari                                  +0.9556 (  +160.0%)
cat__developer_Al Hamra                              +0.8598 (  +136.3%)

--- the 8 that push price DOWN most ---
num__km_to_burj_khalifa           

In [28]:
coefs

,0
num__bedrooms,0.090823
num__bathrooms,0.105417
num__built_up_area_sqft,0.083503
num__plot_area_sqft,-0.090812
num__floor_number,0.079017
...,...
cat__listed_by_Developer,0.005132
cat__listed_by_Owner,-0.002459
cat__agency_tier_Boutique,-0.006333
cat__agency_tier_Large,0.005534


In [29]:
from scipy import stats
residuals = y_test - pred_log

fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))

axes[0].scatter(pred_log, residuals, alpha=0.3, s=12)
axes[0].axhline(0, color="crimson", linestyle="--")
axes[0].set(title="residuals vs predicted", xlabel="predicted log price", ylabel="residual")

axes[1].hist(residuals, bins=45, color="#2b6cb0")
axes[1].axvline(0, color="crimson", linestyle="--")
axes[1].set(title="residual distribution", xlabel="residual")

stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot vs normal")

plt.tight_layout()
plt.savefig("residuals.png", dpi=110, bbox_inches="tight")
plt.close()
print("\nsaved residuals.png")

print(f"\nresidual mean {residuals.mean():+.4f}(should be near 0)")
print(f"residual std  {residuals.std():.4f}")

worst = comparison.reindex(
comparison["error_pct"].abs().sort_values(ascending=False).index)

print("\n--- the 5 worst misses ---")
print(worst.head(5).round(0).to_string(index=False))



saved residuals.png

residual mean +0.0061(should be near 0)
residual std  0.2280

--- the 5 worst misses ---
 actual_aed  predicted_aed   error_aed  error_pct
  8000000.0     18902776.0  10902776.0      136.0
 26500000.0     56456732.0  29956732.0      113.0
  4902296.0      8674487.0   3772191.0       77.0
  2600000.0      4514176.0   1914176.0       74.0
 40000000.0     10676846.0 -29323154.0      -73.0


In [30]:
test_rows = model_df.loc[X_test.index]
print("\n--- mean absolute residual by emirate ---")
print(pd.Series(np.abs(residuals.values),
      index=test_rows["emirate"].values)
      .groupby(level=0).agg(["count", "mean"]).round(3).to_string())


--- mean absolute residual by emirate ---
                count   mean
Abu Dhabi          94  0.137
Ajman              27  0.201
Dubai             426  0.181
Ras Al Khaimah      4  0.142
Sharjah            49  0.172


**Value one property**

In [31]:
new_listing = X_test.iloc[[0]].copy()
estimate = float(np.expm1(model.predict(new_listing))[0])
actual = float(aed_test.iloc[0])
row = model_df.loc[new_listing.index[0]]
new_listing = X_test.iloc[[0]].copy()

print("\n" + "=" * 72)
print("VALUING ONE PROPERTY")
print("=" * 72)

print(f"{int(row['bedrooms'])}-bed {row['property_type']} in {row['area']}")
print(f"{row['built_up_area_sqft']:,.0f} sqft, {row['view']}, {row['furnishing']}")
print(f"\n  model estimate : AED {estimate:,.0f}")
print(f"  actual asking  : AED {actual:,.0f}")
print(f"difference : {100 * (estimate - actual) / actual:+.1f}%")

lo = np.expm1(np.log1p(estimate) - rmse_log)
hi = np.expm1(np.log1p(estimate) + rmse_log)
rmse_log = np.sqrt(mean_squared_error(y_test, pred_log))

print(f"\n honest range  : AED {lo:,.0f} - {hi:,.0f}")
print(" (one RMSE either side - never quote a single number to a client)")


VALUING ONE PROPERTY
2-bed Apartment in Dubai Studio City
1,103 sqft, Pool View, Unfurnished

  model estimate : AED 1,100,715
  actual asking  : AED 1,000,000
difference : +10.1%

 honest range  : AED 876,362 - 1,382,504
 (one RMSE either side - never quote a single number to a client)
